# Install & Imports

In [1]:
# Install torchinfo for model summary
!pip install -q torchinfo

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchinfo import summary

import numpy as np
import cv2
import os
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time
import psutil  # Check RAM usage
import matplotlib.pyplot as plt

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device set to: {device}")

✅ Device set to: cuda


# The Feature Logic (The 4 Boxes)
*This class implements the math for Color, FFT, and Gradients.*

In [2]:
class FeatureExtractor:
    """
    Implements the core logic from 'Idea 1':
    Combining RGB + Color Stats + FFT + Gradients
    """

    @staticmethod
    def get_color_diff(img_np):
        """
        [CYAN BOX] Color Distribution Difference
        Logic: AI images often have 'too perfect' color transitions.
        We quantize the image (reduce colors) and find the difference.
        """
        # Simulate 4-bit quantization (Idea from 'Secret Lies in Color')
        quantized = (img_np // 16) * 16
        diff = np.abs(img_np - quantized)
        return diff / 255.0  # Normalize 0-1

    @staticmethod
    def get_fft_spectrum(img_np):
        """
        [PURPLE BOX] Frequency Domain Information
        Logic: GANs/Diffusion leave checkerboard patterns visible in FFT.
        """
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        f = np.fft.fft2(gray)
        fshift = np.fft.fftshift(f)
        # Log scale magnitude
        magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-10)
        # Min-Max Normalize
        norm_mag = (magnitude_spectrum - magnitude_spectrum.min()) / (magnitude_spectrum.max() - magnitude_spectrum.min())
        return norm_mag

    @staticmethod
    def get_gradients(img_np):
        """
        [GREEN BOX] Luminance Gradients (Physics)
        Logic: Calculates change in light (Gx) and texture (Gy).
        """
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        # Sobel operators
        gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        
        # Normalize
        gx = (gx - gx.min()) / (gx.max() - gx.min() + 1e-8)
        gy = (gy - gy.min()) / (gy.max() - gy.min() + 1e-8)
        return gx, gy

# The Dataset (The Fusion)

*This reads the files and stacks the features together.*

In [3]:
class MultiStreamCIFAKE(Dataset):
    def __init__(self, root_dir, split='train'):
        """
        Kaggle CIFAKE Structure:
        /input/cifake-real-and-ai-generated-synthetic-images/train/REAL
        /input/cifake-real-and-ai-generated-synthetic-images/train/FAKE
        """
        self.root_dir = root_dir
        self.image_paths = []
        self.labels = []
        
        # Define paths based on split
        base_path = os.path.join(root_dir, split)
        classes = {'REAL': 0, 'FAKE': 1}
        
        print(f"Loading {split} data...")
        for cls_name, cls_idx in classes.items():
            cls_path = os.path.join(base_path, cls_name)
            if os.path.exists(cls_path):
                files = os.listdir(cls_path)
                # Let's use a subset for faster demonstration if needed
                # Remove [:2000] to use full dataset
                for fname in files: 
                    self.image_paths.append(os.path.join(cls_path, fname))
                    self.labels.append(cls_idx)
        print(f"✅ Loaded {len(self.image_paths)} images for {split}.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img_pil = Image.open(path).convert('RGB')
        
        # Resize to 224x224 (Standard for MobileNet)
        img_pil = img_pil.resize((224, 224))
        img_np = np.array(img_pil)

        # --- EXTRACT FEATURES ---
        # 1. [ORANGE] RGB (3 ch)
        rgb = img_np / 255.0
        
        # 2. [CYAN] Color Diff (3 ch)
        cdiff = FeatureExtractor.get_color_diff(img_np)
        
        # 3. [PURPLE] FFT (1 ch)
        fft = FeatureExtractor.get_fft_spectrum(img_np)
        fft = np.expand_dims(fft, axis=-1)
        
        # 4. [GREEN] Gradients (2 ch)
        gx, gy = FeatureExtractor.get_gradients(img_np)
        gx = np.expand_dims(gx, axis=-1)
        gy = np.expand_dims(gy, axis=-1)

        # --- [CHECKERED BAR] CONCATENATION ---
        # Total Channels: 3 + 3 + 1 + 2 = 9 Channels
        combined = np.concatenate([rgb, cdiff, fft, gx, gy], axis=-1)
        
        # Convert to Tensor (C, H, W)
        tensor = torch.from_numpy(combined).float().permute(2, 0, 1)
        
        return tensor, self.labels[idx]

# The Lightweight Architecture
*Modifying MobileNetV3-Small.*

In [4]:
def build_idea1_model():
    """
    [YELLOW BOX] Lightweight Pretrained Model
    Using: MobileNetV3-Small (Only ~2.5M params!)
    """
    # Load Pretrained weights (Transfer Learning)
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    
    # --- MODIFY INPUT LAYER ---
    # The original model expects 3 channels. We have 9.
    original_layer = model.features[0][0]
    
    # Create new layer with 9 input channels
    new_layer = nn.Conv2d(
        in_channels=9, # RGB(3)+Color(3)+FFT(1)+Grad(2)
        out_channels=original_layer.out_channels,
        kernel_size=original_layer.kernel_size,
        stride=original_layer.stride,
        padding=original_layer.padding,
        bias=False
    )
    
    # Keep trained weights for RGB, init others
    with torch.no_grad():
        new_layer.weight[:, :3, :, :] = original_layer.weight # Copy RGB weights
        nn.init.kaiming_normal_(new_layer.weight[:, 3:, :, :]) # Random init for new features

    model.features[0][0] = new_layer
    
    # --- MODIFY OUTPUT LAYER ---
    # Change classifier from 1000 classes to 2 (Real vs Fake)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, 2)
    
    return model

model = build_idea1_model().to(device)

# Print Stats to prove "Lightweight"
summary(model, input_size=(1, 9, 224, 224))

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 137MB/s]


Layer (type:depth-idx)                             Output Shape              Param #
MobileNetV3                                        [1, 2]                    --
├─Sequential: 1-1                                  [1, 576, 7, 7]            --
│    └─Conv2dNormActivation: 2-1                   [1, 16, 112, 112]         --
│    │    └─Conv2d: 3-1                            [1, 16, 112, 112]         1,296
│    │    └─BatchNorm2d: 3-2                       [1, 16, 112, 112]         32
│    │    └─Hardswish: 3-3                         [1, 16, 112, 112]         --
│    └─InvertedResidual: 2-2                       [1, 16, 56, 56]           --
│    │    └─Sequential: 3-4                        [1, 16, 56, 56]           744
│    └─InvertedResidual: 2-3                       [1, 24, 28, 28]           --
│    │    └─Sequential: 3-5                        [1, 24, 28, 28]           3,864
│    └─InvertedResidual: 2-4                       [1, 24, 28, 28]           --
│    │    └─Sequential: 3-6 

# Training & Comprehensive Metrics

In [5]:
# --- BLOCK 5: TRAINING & EXECUTION (FIXED) ---

# --- CONFIG ---
BATCH_SIZE = 32 
LR = 0.0005     
EPOCHS = 5      

# --- LOAD DATA ---
# Ensure path is correct
KAGGLE_PATH = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"

print("🔄 Initializing Data Loaders...")
train_ds = MultiStreamCIFAKE(KAGGLE_PATH, split='train')
test_ds = MultiStreamCIFAKE(KAGGLE_PATH, split='test')

# !!! FIX IS HERE: num_workers=0 !!!
# This runs data loading in the main process to avoid freezing
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# --- TRAINING FUNCTION ---
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print("\n🚀 Starting Training (Watch for Batch Progress)...")

for epoch in range(EPOCHS):
    model.train()
    run_loss = 0
    start_t = time.time()
    
    # Added 'enumerate' to show progress
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        run_loss += loss.item()
        
        # PRINT PROGRESS every 100 batches so you know it's not stuck
        if i % 100 == 0:
            print(f"  > Epoch {epoch+1} | Batch {i}/{len(train_loader)} processed...")
        
    epoch_time = time.time() - start_t
    print(f"✅ Epoch {epoch+1} Complete | Loss: {run_loss/len(train_loader):.4f} | Time: {epoch_time:.1f}s")

print("\n💾 Training Complete. Saving Model...")
torch.save(model.state_dict(), "idea1_mobilenet_fusion.pth")

# --- EVALUATION (ALL METRICS) ---
model.eval()
y_true = []
y_pred = []
latencies = []

# Monitor Memory
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / 1024**2 # MB

print("\n📊 Running Detailed Evaluation...")
with torch.no_grad():
    for i, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        
        # Latency Check
        t0 = time.time()
        outputs = model(inputs)
        t1 = time.time()
        latencies.append((t1-t0)/inputs.size(0)) # Time per image
        
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        
        if i % 100 == 0:
             print(f"  > Eval Batch {i}/{len(test_loader)}...")

mem_after = process.memory_info().rss / 1024**2 # MB

# --- PRINT REPORT ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"\n🏆 FINAL RESULTS (Idea 1 Validation)")
print(f"------------------------------------")
print(f"Model:          MobileNetV3-Small (Multi-Stream)")
print(f"Total Params:   {sum(p.numel() for p in model.parameters()):,}")
print(f"Accuracy:       {acc*100:.2f}%")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec:.4f}")
print(f"F1 Score:       {f1:.4f}")
print(f"Avg Latency:    {np.mean(latencies)*1000:.2f} ms per image")
print(f"RAM Usage:      {mem_after - mem_before:.2f} MB used during eval")
print(f"------------------------------------")
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

🔄 Initializing Data Loaders...
Loading train data...
✅ Loaded 100000 images for train.
Loading test data...
✅ Loaded 20000 images for test.

🚀 Starting Training (Watch for Batch Progress)...
  > Epoch 1 | Batch 0/3125 processed...
  > Epoch 1 | Batch 100/3125 processed...
  > Epoch 1 | Batch 200/3125 processed...
  > Epoch 1 | Batch 300/3125 processed...
  > Epoch 1 | Batch 400/3125 processed...
  > Epoch 1 | Batch 500/3125 processed...
  > Epoch 1 | Batch 600/3125 processed...
  > Epoch 1 | Batch 700/3125 processed...
  > Epoch 1 | Batch 800/3125 processed...
  > Epoch 1 | Batch 900/3125 processed...
  > Epoch 1 | Batch 1000/3125 processed...
  > Epoch 1 | Batch 1100/3125 processed...
  > Epoch 1 | Batch 1200/3125 processed...
  > Epoch 1 | Batch 1300/3125 processed...
  > Epoch 1 | Batch 1400/3125 processed...
  > Epoch 1 | Batch 1500/3125 processed...
  > Epoch 1 | Batch 1600/3125 processed...
  > Epoch 1 | Batch 1700/3125 processed...
  > Epoch 1 | Batch 1800/3125 processed...
  > 